In [ ]:
# Import packages 
import os
import sys
workdir_ = '/glade/work/juliob'
if ( workdir_ not in sys.path ):
    sys.path.append(workdir_)
    print( f" a path to {workdir_} added in {__name__} ")

import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature




from myPythonTools.TopoAna import Plots as Po
from myPythonTools.TopoAna import rdtilemap as Ti
from myPythonTools.TopoAna import rdremap as RM
from myPythonTools.TopoAna import rdridgelist as RL


import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.tri as tri
import matplotlib.colors as colors

from scipy.io import FortranFile

import importlib
import glob
import copy

importlib.reload(Po)
importlib.reload(Ti)
importlib.reload(RM)
importlib.reload(RL)

#sys.path.append('../Plotting/')
#   Now you can imprt modules in ../Plotting
#import xyp_plot as xyp
#import ana as a


In [ ]:
fileN ='/glade/derecho/scratch/juliob/archive/GW_UnitTest/ERA5-era5test05/ERA5-era5test05.h.2014-01-15-00000.nc'
X=xr.open_dataset( fileN )

In [ ]:
X.bnd_topo


In [ ]:
tau_a = X.TAU_A_RDG.values
tau_b = X.TAU_B_RDG.values
taudsw = X.TAUDSW.values
tauoro = X.TAUORO.values

sgh=X.SGH.values
lonc=X.lon.values
latc=X.lat.values

usrc = X.USRC_RDG.values
vsrc = X.VSRC_RDG.values


# ---------------------------------
# Basic wave-related quantities
# consistent with rdg GW scheme.
#----------------------------------
nsrc=X.NSRC_RDG.values
ubsrc=X.UBMSRC_RDG.values
hwdth=1000.*X.HWDTH_RDG.values 

ktild=1./(hwdth+1.)
rho=1.0
delprm2 = hwdth*taudsw /(rho*nsrc*ubsrc)
delprm = np.sqrt( delprm2 )
uprm = nsrc*delprm
warmy=delprm*9.8/1000.

In [ ]:
#co = plt.tricontourf( lonc,latc, np.sum( tau_b-0*tau_a, axis=(0,1,2) )[:] )
co = plt.tricontourf( lonc,latc, np.sum( taudsw - tauoro, axis=(0,1) )[:], cmap='plasma', levels=np.linspace(0,10,11) )
li=plt.tricontour( lonc,latc, sgh, levels=[-1,0,1,10000] ,colors='w' )
plt.xlim(240,250)
plt.ylim(30,36)
plt.xlabel('Lon')
plt.ylabel('Lat')
plt.title( 'Jan 15 2014 ' )
plt.colorbar(co)

In [ ]:
print( np.asarray([240,250])-360)


In [ ]:
TopoRoot = '/glade/work/juliob/Topo/NCARTopoJTB-fire/'     #cases/ne30pg3_Sco100_small_rdgs/
#Case = 'ne30pg3_Sco100_Sfi6_v11_test'
#Case = 'ne30pg3_Sco100_v8_test'
Case = 'ne30pg3_Sco100_Sfi6_v200_rch1p0_e02'
Case = 'ne30pg3_Sco100_Rw050_v200_rch1p0_e02'
Case = 'ne30pg3_Sco100_v202_rch1p0_e01'
ogrid= 'ne30pg3'



In [ ]:
import glob
import os
import xarray as xr
import numpy as np

outdir = TopoRoot + 'cases/' + Case + '/output'

# Find ordinary files, explicitly excluding *_tiles.nc
foo = [
    f for f in glob.glob(outdir + '/'+ ogrid + '*.nc')
    if not f.endswith('_tiles.nc')
]

# Take newest file
FileN = max(foo, key=os.path.getmtime)

# Corresponding tiles file
FileTiles = FileN.replace('.nc', '_tiles.nc')

if not os.path.exists(FileTiles):
    raise FileNotFoundError(f"No corresponding tiles file:\n{FileTiles}")

print("Main file: ", FileN)
print("Tiles file:", FileTiles)
print("Topo file: ", X.bnd_topo )

# Main grid file
dCAM = xr.open_dataset(FileN)

print(list(dCAM.variables))
print(np.shape(dCAM['lat']))

target_lon  = dCAM['lon'].values
target_lat  = dCAM['lat'].values
target_topo = dCAM['PHIS'].values / 9.8

# Tile file
dTiles = xr.open_dataset(FileTiles)

print(list(dTiles.variables))

In [ ]:
import paint_tiles_nc as Pti
importlib.reload( Pti )


from paint_tiles_nc import load_tiles_nc, paint_tiles_nc

filn = '/glade/work/juliob/Topo/NCARTopoJTB-fire//cases/ne30pg3_Sco100_v202_rch1p0_e01/output/ne30pg3_gmted2010_modis_bedmachine_nc3000_Laplace0100_noleak_greenlndantarcsgh30fac2.50_20260907_tiles.nc'

tg = Pti.load_tiles_nc( filn, ncube=3000, grid_length_scale=3.3)
A  = (taudsw - tauoro)[0,:,:] # (ncol, nrdg)
A  = warmy[0,:,:] # (ncol, nrdg)

field = Pti.paint_tiles_nc(A, tg, shape='rectangle',  width_scale=1.0, verbose=True, ir_range=(0,43) ,  side='downwind', usrc=usrc[0,:,:], vsrc=vsrc[0,:,:]   , modulation='ramp_downwind')
field2 = Pti.paint_tiles_nc(mxd, tg, shape='rectangle',  width_scale=0.1, verbose=True, ir_range=(0,43) )


In [ ]:
plt.plot( field[0,:,:].flatten() )

In [ ]:
mxd=dTiles.MXDIS

mxd.shape

In [ ]:
field.shape

In [ ]:

fig,ax=plt.subplots(figsize=(7,5) )

#plt.contourf( field[3,:,:] , cmap='plasma', levels= np.linspace(0.1,1.1,11) ,alpha=0.8)
#ax.contourf( field2[3,:,:] , colors='black', levels= 51 )
c1 = ax.contourf( field[3,:,:] , cmap='OrRd', levels= np.linspace(1,31,41) ,alpha=1)
ax.set_xlim( 500,700 )
ax.set_ylim( 2600,2800 )
plt.colorbar( c1 )

In [ ]:
taudsw.shape

In [ ]:
plt.plot( taudsw[0,0,:] )
plt.plot( tauoro[0,0,:] )


In [ ]:
delprm = 

In [ ]:
plt.plot( warmy[0,0,:] ,'.')

In [ ]:
plt.plot( ubsrc[0,0,:] ,'.')



uprm=hwdth * np.sqrt( taudsw ) / ubsrc 
delprm=hwdth * np.sqrt( taudsw ) / (ubsrc *nsrc)

In [ ]:
plt.plot((uprm+ubsrc)[0,0,:] )
plt.plot((uprm)[0,0,:] )
utot=uprm+ubsrc

In [ ]:
#plt.plot((9.8/1000.)*(delprm)[0,0,:] ) 
warmy = (9.8/1000.)*delprm
plt.plot(warmy[0,0,:] ) 

In [ ]:
X

In [ ]:
from regrid_to_cube import read_cube_target_map, regrid_to_cube

itrgt = read_cube_target_map('/glade/work/juliob/Topo/NCARTopoJTB-fire/cases/ne30pg3_Sco100_v202_rch1p0_e01/output/grid_remap_nc3000_ne30pg3.dat', verbose=True)
#cube  = regrid_to_cube(taudsw_column_sum, itrgt)   # -> (6, 3000, 3000)

In [ ]:
te=X.T.values
te_cube  = regrid_to_cube(te[0,92,:], itrgt)  



In [ ]:
plt.contourf( te_cube[3,:,:] )

In [ ]:
import numpy as np
from paint_tiles_nc import load_tiles_nc, paint_tiles_nc

tg = load_tiles_nc('..._tiles.nc', ncube=3000, grid_length_scale=3.336)

def km_decay(s, t, L, W):
    """Exponential decay with a 25 km e-folding scale."""
    return np.exp(-np.abs(t) * 3.336 / 25.0)

field = paint_tiles_nc(A, tg, shape='triangle', side='downwind',
                       usrc=usrc, vsrc=vsrc, modulation=km_decay)